# Main figures

Figs. 2 and 3 of the paper, from the outputs of the pipeline (`../workflow/`). Set the paths in the first cell and run it, then run the cells of either figure on their own. Each figure is saved as a PDF to `FIG_DIR`.

In [ ]:
# =============================================================================
# Paths: replace with yours
#   RESULTS   results_dir of config/config.yaml: the outputs of the pipeline
#   DATA_DIR  spectra and signatures of other studies (README.md)
#   FIG_DIR   where the figures are saved
# =============================================================================
RESULTS  = "../results"
DATA_DIR = "data"
FIG_DIR  = "../figures"

CAND = f"{RESULTS}/somatic_mutation_candidates"   # somatic mutations, callable bps, trinucleotides
ACC  = f"{RESULTS}/chromatin_accessibility"       # accessible chromatin and its callable bps

## Fig. 2. Somatic point mutations as a function of age

The first cell loads the somatic mutations and callable bps of each animal and defines the plotting functions; the second draws the figure.

In [ ]:
# =============================================================================
# Fig. 2: data and plotting functions
#
# ours  per animal (the libraries of one animal summed): somatic mutations
#       (SINGLETONS_ON_MOLECULE <= 2, minimum untrimmed coverage 10), callable bps,
#       and point mutations per diploid genome = mutations / callable bps x 2 x the
#       length of the T2T autosomes
# Fit   ordinary least squares on the animals, with the 95% confidence band of the line
# =============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle
from matplotlib import legend as mpl_legend
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, MaxNLocator, MultipleLocator

plt.rcParams["pdf.fonttype"] = 42           # TrueType fonts in the PDFs

# ------------------------------ data knobs -----------------------------------
AGE_MAX_SPM  = 2                         # <= this many singletons per molecule
AGE_FLOOR    = 10                        # minimum untrimmed coverage of the mutations and callable bps

# ------------------------------ style knobs ----------------------------------
C_OURS       = "#4C72B0"                 # blue circles
MARKER_SIZE  = 5
BAND_ALPHA   = 0.15                      # 95% band tint
EXT_LS       = (0, (4, 2.5))             # line from the youngest animals back to age 0: dashed
LINE_LW      = 1.2
LABEL_FONT, TICK_FONT, LEGEND_FONT = 9, 8, 8   # points
AXIS_LW, TICK_LEN = 0.8, 3
XLABEL       = "Age (weeks)"
XLIM         = (0, 74)                   # weeks
Y_PAD        = 0.05                      # y spans everything drawn, plus this share of it beyond each end
Y_NBINS      = 6                         # at most this many y tick intervals
FIT_LOC      = "upper left"              # where the fit's legend sits
LOW_IN       = 0.17                      # broken y axis: height of the strip holding the 0 tick, inches
GAP_IN       = 0.06                      # gap between the strip and the main panel, inches
BREAK_IN     = 0.045                     # half-length of each break mark on the left spine, inches
LEGEND_CLEAR_IN = 0.04                   # what is drawn under the legend sits this far below it, inches
R2_GAP_PT    = 2                         # R2, if drawn, sits this far under the legend icon, points

WEEKS_PER_YEAR = 365 / 7

# ------------------------------- data ----------------------------------------
age_calls = pd.read_csv(f"{CAND}/somatic_mutation_candidates.tsv", sep="\t",
                        usecols=["LIBRARY", "ANIMAL", "SINGLETONS_ON_MOLECULE"])
age_calls = age_calls[age_calls.SINGLETONS_ON_MOLECULE <= AGE_MAX_SPM]
_cb = pd.read_csv(f"{CAND}/callable_bases.tsv", sep="\t")
_cb = _cb[_cb.floor == AGE_FLOOR].set_index("sample")
T2T_AUTOSOME_BP = int(_cb.autosome_bp.iloc[0])                # length of T2T1..T2T19, per haploid genome
_lib2animal = age_calls.groupby("LIBRARY").ANIMAL.first()
if set(_lib2animal.index) != set(_cb.index):
    raise ValueError("libraries differ between the candidate table and callable_bases.tsv")

ours = pd.DataFrame({"mutations": age_calls.groupby("ANIMAL").size(),
                     "callable_depth": _cb.callable_depth.groupby(_lib2animal).sum()})
ours["age_wk"] = ours.index.str.extract(r"^(\d+)wk", expand=False).astype(int).to_numpy()
ours["rate_plain"] = ours.mutations / ours.callable_depth
ours["per_genome_plain"] = ours.rate_plain * 2 * T2T_AUTOSOME_BP
ours = ours.sort_values("age_wk")


# ------------------------------- fit -----------------------------------------
class Fit:
    """OLS of y on x: line, 95% band of the line, slope's 95% CI and P (slope = 0)."""
    def __init__(self, x, y):
        self.x = np.asarray(x, float)
        self.res = sm.OLS(np.asarray(y, float), sm.add_constant(self.x)).fit()
        (self.b0, self.b1), self.ci = self.res.params, self.res.conf_int(0.05)
        self.p = self.res.pvalues[1]
    def line(self, xs):
        return self.b0 + self.b1 * np.asarray(xs, float)
    def band(self, xs):
        return self.res.get_prediction(sm.add_constant(np.asarray(xs, float),
                                                       has_constant="add")).conf_int(alpha=0.05)


INSIDE = np.linspace(ours.age_wk.min(), ours.age_wk.max(), 200)   # the ages of the animals


# ------------------------------- drawing -------------------------------------
def tint(color, alpha):
    """color laid over white at this opacity, as an opaque colour."""
    return tuple(1 - alpha * (1 - c) for c in to_rgb(color))


class FitKey:
    """Legend key of the fit: the 95% band as a shaded box with the line through
    its middle. Use: ax.legend([FitKey()], [label], handler_map={FitKey: FitKey()})."""
    def legend_artist(self, legend, orig_handle, fontsize, handlebox):
        x0, y0 = -handlebox.xdescent, -handlebox.ydescent
        w, h = handlebox.width, handlebox.height
        box = Rectangle((x0, y0), w, h, facecolor=tint(C_OURS, BAND_ALPHA), lw=0,
                        transform=handlebox.get_transform())
        line = plt.Line2D([x0, x0 + w], [y0 + h / 2] * 2, color=C_OURS, lw=LINE_LW,
                          solid_capstyle="butt", transform=handlebox.get_transform())
        handlebox.add_artist(box)
        handlebox.add_artist(line)
        return box


def p_text(p):
    """p as 'm × 10^e' (mathtext), or plain from 0.001 up."""
    if p >= 1e-3:
        return f"{p:.2g}"
    m, e = f"{p:.1e}".split("e")
    return rf"{m} × 10$^{{{int(e)}}}$"


def fit_legend(ax, fit, label_fmt, unit=1.0, extra=()):
    """The fit's legend: its key, labelled with the slope per year ({slope}, {lo}, {hi},
    in units of `unit`) and the slope's P ({p}). `extra`: (handle, label) pairs listed
    above the fit's key. Returns the legend."""
    k = WEEKS_PER_YEAR / unit
    label = label_fmt.format(slope=fit.b1 * k, lo=fit.ci[1][0] * k, hi=fit.ci[1][1] * k, p=p_text(fit.p))
    return ax.legend([h for h, _ in extra] + [FitKey()], [t for _, t in extra] + [label],
                     handler_map={FitKey: FitKey()}, handlelength=2.6, handleheight=1.6,
                     loc=FIT_LOC, frameon=False, fontsize=LEGEND_FONT)


def draw_age(ax, fit, y, unit=1.0, xlim=XLIM):
    """The animals (points), the fit and its 95% band, y in units of `unit`; the fit
    runs dashed (line only) from the youngest animals back to xlim[0] (age 0).
    Returns the y range drawn and the points."""
    before = np.linspace(xlim[0], INSIDE[0], 100)
    xs = INSIDE
    ax.plot(before, fit.line(before) / unit, color=C_OURS, lw=LINE_LW, ls=EXT_LS, zorder=2)
    b = fit.band(INSIDE) / unit
    ax.fill_between(INSIDE, b[:, 0], b[:, 1], color=tint(C_OURS, BAND_ALPHA), lw=0, zorder=1)
    ax.plot(INSIDE, fit.line(INSIDE) / unit, color=C_OURS, lw=LINE_LW, zorder=2)
    pts, = ax.plot(ours.age_wk, np.asarray(y) / unit, ls="", marker="o", ms=MARKER_SIZE, color=C_OURS,
                   mew=0, zorder=4)
    lo = [np.min(y), fit.band(xs)[:, 0].min(), fit.line(xlim[0])]
    hi = [np.max(y), fit.band(xs)[:, 1].max()]
    return min(lo) / unit, max(hi) / unit, pts


def style_age(ax, ylabel=None, xlabel=True, xlim=XLIM):
    """Fonts, ticks and spine widths."""
    ax.set_xlim(xlim)
    if xlabel:
        ax.set_xlabel(XLABEL, fontsize=LABEL_FONT)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=LABEL_FONT)
    ax.tick_params(labelsize=TICK_FONT, length=TICK_LEN, width=AXIS_LW)
    for s in ax.spines.values():
        s.set_linewidth(AXIS_LW)


def broken_age_axes(fig, box_in, low_in=LOW_IN, gap_in=GAP_IN):
    """A panel with a broken y axis, filling box_in = (x0, y0, width, height) in inches
    from the figure's lower left: the main panel on top, a strip low_in tall at the
    bottom that holds the 0 tick, and gap_in between them. Returns (main panel, strip).
    Draw on the main panel, then call break_y."""
    W, H = fig.get_size_inches()
    x0, y0, w, h = box_in
    main_in = h - low_in - gap_in
    assert main_in > 1, f"the panel box {box_in} leaves too little room for the main panel"
    ax = fig.add_axes([x0 / W, (y0 + low_in + gap_in) / H, w / W, main_in / H])
    axl = fig.add_axes([x0 / W, y0 / H, w / W, low_in / H], sharex=ax)
    return ax, axl


def break_y(ax, axl, lo, hi, y_ticks=None, xlim=XLIM, break_in=BREAK_IN, y_break=None, y_top=None):
    """Finish a panel made by broken_age_axes: the main panel from y_break (None: Y_PAD
    below lo) to y_top (None: Y_PAD above hi), ticks at y_ticks (None: automatic); the
    strip shows 0 at the same units per inch. Break marks cross the left spine; the
    right spine runs unbroken from the bottom of the strip to the top of the panel."""
    fig = ax.figure
    W, H = fig.get_size_inches()
    pm, ps = ax.get_position(), axl.get_position()
    main_in, low_in, w_in = pm.height * H, ps.height * H, pm.width * W
    y0 = lo - Y_PAD * (hi - lo) if y_break is None else y_break
    y1 = hi + Y_PAD * (hi - lo) if y_top is None else y_top
    assert y0 < lo and y1 > hi, \
        f"the main panel ({y0:.3g} to {y1:.3g}) must hold everything drawn ({lo:.3g} to {hi:.3g})"
    ax.set_ylim(y0, y1)
    ax.yaxis.set_major_locator(FixedLocator(y_ticks) if y_ticks is not None
                               else MaxNLocator(nbins=Y_NBINS, steps=[1, 2, 2.5, 5, 10]))
    axl.set_ylim(0, (y1 - y0) / main_in * low_in)
    axl.yaxis.set_major_locator(FixedLocator([0]))
    style_age(ax, xlabel=False, xlim=xlim)
    style_age(axl, xlim=xlim)
    for a, side in ((ax, "bottom"), (ax, "right"), (axl, "top"), (axl, "right")):
        a.spines[side].set_visible(False)
    ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    fig.add_artist(Line2D([pm.x1, pm.x1], [ps.y0, pm.y1], transform=fig.transFigure, color="black",
                          lw=AXIS_LW, solid_capstyle="projecting"))      # the right spine
    for a, yb, h_in in ((ax, 0.0, main_in), (axl, 1.0, low_in)):     # break marks on the left spine
        dx, dy = break_in / w_in, break_in / h_in
        a.plot([-dx, dx], [yb - dy, yb + dy], transform=a.transAxes, color="black",
               lw=AXIS_LW, clip_on=False, zorder=6, solid_capstyle="butt")


def ylabel_beside(fig, axes, text, fontsize=LABEL_FONT):
    """One y label for a stack of panels, centred on their joint height, 4 pt left of
    the leftmost tick label."""
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    x0 = min(t.get_window_extent(rend).x0 for a in axes for t in a.get_yticklabels() if t.get_text())
    y_lo, y_hi = min(a.get_position().y0 for a in axes), max(a.get_position().y1 for a in axes)
    lab = fig.text((x0 - 4 * fig.dpi / 72) / fig.bbox.width, (y_lo + y_hi) / 2, text,
                   fontsize=fontsize, rotation=90, ha="right", va="center")
    assert lab.get_window_extent(rend).x0 >= 0, "the y label runs off the figure -- widen the margins or FIGSIZE"
    return lab


def r2_under_legend(ax, fit, legend, fmt="R² = {r2:.2f}", gap_pt=R2_GAP_PT):
    """The fit's R2 in black, left-aligned with the legend's fit icon, gap_pt under it."""
    icon = legend.legend_handles[-1]
    return ax.annotate(fmt.format(r2=fit.res.rsquared), xy=(0, 0), xycoords=icon,
                       xytext=(0, -gap_pt), textcoords="offset points", ha="left", va="top",
                       fontsize=LEGEND_FONT, color="black", zorder=6)


def drawn_edges(fit, y, xlim=XLIM):
    """(x, y, pad_in) samples of the top and of the bottom of everything draw_age draws;
    pad_in is how far the mark reaches beyond (x, y), in inches. Returns (tops, bottoms)."""
    r_in, l_in = MARKER_SIZE / 2 / 72, LINE_LW / 2 / 72
    xs_line = np.linspace(xlim[0], INSIDE[-1], 400)
    xs_band = INSIDE
    band = fit.band(xs_band)
    x_pts, y_pts = ours.age_wk.to_numpy(float), np.asarray(y, float)
    tops = [(xs_line, fit.line(xs_line), l_in), (xs_band, band[:, 1], 0.0), (x_pts, y_pts, r_in)]
    bottoms = [(xs_line, fit.line(xs_line), l_in), (xs_band, band[:, 0], 0.0), (x_pts, y_pts, r_in)]
    return tops, bottoms


def clear_top(ax, y0, tops, artists, clear_in=LEGEND_CLEAR_IN):
    """The lowest y top of the main panel (its bottom at y0) at which every sample in
    `tops` (drawn_edges) under one of `artists` sits clear_in below it. Set the x
    limits first."""
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    bb = ax.get_window_extent(rend)
    main_in, x_per_in = bb.height / fig.dpi, np.diff(ax.get_xlim())[0] / (bb.width / fig.dpi)
    top = -np.inf
    for art in artists:
        a = art.get_window_extent(rend)
        depth_in = (bb.y1 - a.y0) / fig.dpi
        x_lo, x_hi = ax.transData.inverted().transform([(a.x0, bb.y0), (a.x1, bb.y0)])[:, 0]
        for x, y, pad_in in tops:
            x, y = np.asarray(x, float), np.asarray(y, float)
            reach = (pad_in + clear_in) * x_per_in
            sel = (x >= x_lo - reach) & (x <= x_hi + reach)
            if sel.any():
                g = (depth_in + clear_in + pad_in) / main_in
                top = max(top, (y[sel].max() - g * y0) / (1 - g))
    return top


def check_clear(ax, artist, edges, above, clear_in=LEGEND_CLEAR_IN * 0.99):
    """Nothing drawn (edges from drawn_edges) comes within clear_in of the artist."""
    fig = ax.figure
    fig.canvas.draw()
    a = artist.get_window_extent(fig.canvas.get_renderer())
    for x, y, pad_in in edges:
        pad = (pad_in + clear_in) * fig.dpi
        p = ax.transData.transform(np.column_stack([np.asarray(x, float), np.asarray(y, float)]))
        sel = (p[:, 0] >= a.x0 - pad) & (p[:, 0] <= a.x1 + pad)
        bad = p[sel, 1] + pad > a.y0 if above else p[sel, 1] - pad < a.y1
        assert not bad.any(), "a legend or label touches the data -- change FIGSIZE, Y_BREAK or LEGEND_CLEAR_IN"


def check_text_apart(fig, axes, extra=()):
    """No two pieces of text overlap, and all of them lie inside the figure."""
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    items = list(fig.texts)
    for t in extra:
        if isinstance(t, mpl_legend.Legend) and not t.get_frame_on():
            items += list(t.get_texts()) + list(t.legend_handles)
        else:
            items.append(t)
    for a in axes:
        (x_lo, x_hi), (y_lo, y_hi) = a.get_xlim(), a.get_ylim()
        items += [t for t in a.get_xticklabels() if x_lo <= t.get_position()[0] <= x_hi]
        items += [t for t in a.get_yticklabels() if y_lo <= t.get_position()[1] <= y_hi]
        items += a.texts + [a.xaxis.label, a.yaxis.label]
    seen, boxes = set(), []
    for t in items:
        if id(t) in seen or not t.get_visible() or not getattr(t, "get_text", lambda: "x")():
            continue
        seen.add(id(t))
        boxes.append((getattr(t, "get_text", lambda: type(t).__name__)(), t.get_window_extent(rend)))
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            assert not boxes[i][1].overlaps(boxes[j][1]), \
                f"text overlaps: {boxes[i][0]!r} and {boxes[j][0]!r} -- change FIGSIZE or the knobs"
    for n, b in boxes:
        assert b.x0 >= 0 and b.y0 >= 0 and b.x1 <= fig.bbox.x1 and b.y1 <= fig.bbox.y1, \
            f"{n!r} runs off the figure -- widen the margins or FIGSIZE"


def report_fit(fit, unit, unit_name):
    k = WEEKS_PER_YEAR
    s, lo, hi = fit.b1 * k / unit, fit.ci[1][0] * k / unit, fit.ci[1][1] * k / unit
    print(f"   fit: {s:.3g} [{lo:.3g}, {hi:.3g}] {unit_name} per year, intercept {fit.b0 / unit:.3g}, "
          f"R2 {fit.res.rsquared:.3f}, slope P {fit.p:.3g}")

In [ ]:
# =============================================================================
# Fig. 2: point mutations per diploid genome vs age, the regression line and its
# 95% confidence band; the y axis is broken above 0
# =============================================================================
FIGSIZE   = (3.25, 2.6)                 # width, height in inches
MARGINS   = (0.50, 0.08, 0.08, 0.42)    # inches around the panel: left, right, top, bottom
YLABEL    = "Point mutations per diploid genome"
FIT_LABEL = "slope = {slope:.1f} per year, p = {p}"   # {slope}, {lo}, {hi} per year; {p}
R2_LABEL  = None                        # e.g. "R² = {r2:.2f}": R2 under the legend; None: not drawn
X_STEP    = 10                          # weeks between x ticks
Y_BREAK   = 44                          # the main panel starts here; None: Y_PAD below the data
Y_TICKS   = [50, 70, 90]                # main panel's y ticks; None: automatic
SAVE_AS   = "point_mutations_per_genome_vs_age_uncorrected.pdf"

y_plain = ours.per_genome_plain.to_numpy()
fit_plain = Fit(ours.age_wk, y_plain)

W, H = FIGSIZE
fig = plt.figure(figsize=FIGSIZE)
left, right, top, bottom = MARGINS
ax, axl = broken_age_axes(fig, (left, bottom, W - left - right, H - top - bottom))
lo, hi, _ = draw_age(ax, fit_plain, y_plain, xlim=XLIM)
ax.set_xlim(XLIM)
leg = fit_legend(ax, fit_plain, FIT_LABEL)
under_top = [leg] + ([r2_under_legend(ax, fit_plain, leg, R2_LABEL)] if R2_LABEL else [])

# y range: from Y_BREAK to Y_PAD above the data, higher if the legend needs room
tops, _ = drawn_edges(fit_plain, y_plain, XLIM)
y0 = lo - Y_PAD * (hi - lo) if Y_BREAK is None else Y_BREAK
y_top = max(hi + Y_PAD * (hi - lo), clear_top(ax, y0, tops, under_top))
break_y(ax, axl, lo, hi, Y_TICKS, xlim=XLIM, y_break=Y_BREAK, y_top=y_top)
axl.xaxis.set_major_locator(MultipleLocator(X_STEP))
ylabel_beside(fig, (ax, axl), YLABEL)
for art in under_top:
    check_clear(ax, art, tops, above=True)
check_text_apart(fig, (ax, axl), extra=[leg])

os.makedirs(FIG_DIR, exist_ok=True)
path = os.path.join(FIG_DIR, SAVE_AS)
fig.savefig(path)                                                  # saved at FIGSIZE
plt.show()

print(f"-> {path}")
print(f"{'animal':>17s} {'age':>4s} {'mutations':>9s} {'callable bps':>16s} {'per genome':>11s}")
for a, v in ours.iterrows():
    print(f"{a:>17s} {int(v.age_wk):>4d} {int(v.mutations):>9,} {int(v.callable_depth):>16,} {v.per_genome_plain:>11.1f}")
report_fit(fit_plain, 1.0, "per diploid genome")

## Fig. 3. Spectrum, similarity to other spectra and chromatin accessibility

The first cell builds the spectrum and its cosine similarities (panels a and b), the second the mutation rates in accessible and inaccessible chromatin (panel c), and the third draws the figure.

In [ ]:
# =============================================================================
# Fig. 3a-b: spectrum and cosine similarities
#
# ours        somatic mutations (SINGLETONS_ON_MOLECULE <= 2, minimum untrimmed
#             coverage 10), 96 types per animal; each animal is corrected to a
#             reference genome: counts / w, w = the trinucleotide's share of the
#             animal's callable bps (context_opportunity.tsv) / its share of the
#             reference. The corrected animals are summed.
# references
#   CR        Gronska-Peski et al. 2026, cerebellar neurons of mice fed ad libitum:
#             autosomal mutations per mouse, each mouse corrected with its published
#             w (to mm10); ours corrected to the mm10 autosomes
#   mouse     Riva et al. 2020 mouse signatures (mSBS, mm10); ours corrected to mm10
#   human     COSMIC v3.4 SBS signatures (GRCh37); ours corrected to GRCh37
# fit         for each set of signatures in SIM_MOUSE_SETS / SIM_HUMAN_SETS, the
#             non-negative mix with the highest cosine to ours (non-negative least
#             squares); the set with the highest cosine is kept
# samples     SIM_N_SAMPLES samples of our size drawn from each reference spectrum
#             (signatures: the fitted mix): each animal draws its number of
#             mutations from reference x its w and is corrected as ours; p = share
#             of samples with a cosine at or below ours
# =============================================================================
import os
import numpy as np
import pandas as pd
import SigProfilerAssignment
import SigProfilerMatrixGenerator
from scipy.optimize import nnls

# ------------------------------ knobs ----------------------------------------
SIM_MAX_SPM     = 2                     # <= this many singletons per molecule
SIM_MOUSE_SETS  = [("mSBS1", "mSBS5"), ("mSBS1", "mSBS40")]       # Riva et al. 2020
SIM_HUMAN_SETS  = [("SBS1", "SBS5", "SBS16")]                     # COSMIC v3.4
SIM_N_SAMPLES   = 10_000                # samples of our size from each reference
SIM_SEED        = 0

SIM_CR      = f"{DATA_DIR}/gronska_peski_2026_al_cerebellar_spectra.tsv"
SIM_RIVA    = f"{DATA_DIR}/riva_2020_mSBS_mm10.tsv"
SIM_COSMIC  = os.path.join(os.path.dirname(SigProfilerAssignment.__file__), "data",
                           "Reference_Signatures", "GRCh37", "COSMIC_v3.4_SBS_GRCh37.txt")
SIM_CTX_DIR = os.path.join(os.path.dirname(SigProfilerMatrixGenerator.__file__),
                           "references", "chromosomes", "context_distributions")

# ------------------------------ helpers --------------------------------------
_SUBS = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
SIM_CHANNELS = [f"{a}[{s}]{b}" for s in _SUBS for a in "ACGT" for b in "ACGT"]   # substitution-major
_SIM_CTX = [c[0] + c[2] + c[6] for c in SIM_CHANNELS]                           # channel -> folded triplet
_SIM_RC = str.maketrans("ACGT", "TGCA")
_sim_fold = lambda t: t if t[1] in "CT" else t.translate(_SIM_RC)[::-1]          # TGT -> ACA


def sim_cosine(a, b):
    """Cosine of two spectra, or row by row of two stacks of spectra."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    return (a * b).sum(-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1))


def genome_share(build, scope):
    """Folded 32-trinucleotide share of a SigProfilerMatrixGenerator genome,
    whole (all chromosomes in the table) or autosomes."""
    g = pd.read_csv(f"{SIM_CTX_DIR}/context_counts_{build}_96.csv", index_col=0)
    if scope == "autosomes":
        g = g[[k for k in g.columns if str(k) not in ("X", "Y", "MT", "M")]]
    g = g.sum(axis=1)
    g = g.groupby([_sim_fold(t) for t in g.index]).sum()
    return g / g.sum()


def our_units(anchor):
    """Our animals as (raw counts, correction factor 1 / w), both animal x 96."""
    w = _SIM_SHARE / anchor.reindex(_SIM_SHARE.columns)
    return SIM_RAW.to_numpy(float), 1 / w[_SIM_CTX].to_numpy()


def corrected(units):
    """Corrected spectrum of a set of units: sum of counts x factor."""
    return (units[0] * units[1]).sum(axis=0)


def fit_mix(y, S):
    """Highest-cosine non-negative mix of the columns of S (96 x k) for the spectrum
    y. Returns the cosine and the proportions (summing to 1)."""
    coef, _ = nnls(S, y / y.sum())
    return float(sim_cosine(y, S @ coef)), coef / coef.sum()


def fit_best(y, sigs, sets):
    """fit_mix for every set of signatures; (cosine, proportions, set) of the best."""
    return max(((*fit_mix(y, sigs[list(s)].to_numpy()), tuple(s)) for s in sets), key=lambda f: f[0])


_sim_rng = np.random.default_rng(SIM_SEED)


def sample_from(ref, units, n_rep):
    """n_rep spectra of our size drawn from the reference spectrum `ref`: each animal
    draws its number of mutations from ref x its w, corrected back."""
    raw, fac = units
    out = np.zeros((n_rep, raw.shape[1]))
    for r, f in zip(raw, fac):
        p = ref / f
        out += _sim_rng.multinomial(int(r.sum()), p / p.sum(), size=n_rep) * f
    return out


# ------------------------------- ours ----------------------------------------
_c = pd.read_csv(f"{CAND}/somatic_mutation_candidates.tsv", sep="\t",
                 usecols=["LIBRARY", "ANIMAL", "SINGLETONS_ON_MOLECULE", "SBS96_CHANNEL"])
_c = _c[_c.SINGLETONS_ON_MOLECULE <= SIM_MAX_SPM]
SIM_RAW = _c.groupby(["ANIMAL", "SBS96_CHANNEL"]).size().unstack(fill_value=0)
SIM_RAW = SIM_RAW.reindex(columns=SIM_CHANNELS, fill_value=0)
SIM_N_MUT = int(SIM_RAW.to_numpy().sum())
_opp = pd.read_csv(f"{CAND}/context_opportunity.tsv", sep="\t")
_opp["animal"] = _opp["sample"].map(_c.groupby("LIBRARY").ANIMAL.first())
if _opp.animal.isna().any() or set(_opp.animal) != set(SIM_RAW.index):
    raise ValueError("libraries differ between the candidate table and context_opportunity.tsv")
_SIM_SHARE = _opp.assign(ctx=_opp.triplet.map(_sim_fold)).groupby(["animal", "ctx"]).callable_depth.sum().unstack()
_SIM_SHARE = _SIM_SHARE.div(_SIM_SHARE.sum(axis=1), axis=0).loc[SIM_RAW.index]

_t2t = pd.read_csv(f"{CAND}/t2t_reference_context_counts.tsv", sep="\t")
_t2t = _t2t.groupby(_t2t.triplet.map(_sim_fold)).autosomes.sum()
REFERENCE_SHARE = {"T2T": _t2t / _t2t.sum(),
                   "mm10 autosomes": genome_share("mm10", "autosomes"),
                   "mm10": genome_share("mm10", "whole"),
                   "GRCh37": genome_share("GRCh37", "whole")}
U_OURS = {k: our_units(v) for k, v in REFERENCE_SHARE.items()}
SPECTRUM = corrected(U_OURS["T2T"])                  # panel a: corrected to the T2T autosomes

# ------------------------------- CR ------------------------------------------
_an = pd.read_csv(SIM_CR, sep="\t", index_col=0).reindex(SIM_CHANNELS)
_cols = [k for k in _an.columns if k.startswith("CR_AL|") and not k.endswith("__uncorrected")]
_raw = _an[[k + "__uncorrected" for k in _cols]].to_numpy().T             # mouse x 96, mutations
_cor = _an[_cols].to_numpy().T                                               # mouse x 96, corrected
U_CR = (_raw, np.divide(_cor, _raw, out=np.zeros_like(_cor), where=_raw > 0))

# ------------------------------ signatures -----------------------------------
SIGS = {"mouse": pd.read_csv(SIM_RIVA, sep="\t", index_col=0).reindex(SIM_CHANNELS),
        "human": pd.read_csv(SIM_COSMIC, sep="\t", index_col=0).reindex(SIM_CHANNELS)}

# --------------------------- cosines and samples -----------------------------
COMPARISONS = {   # key: (label, our units, reference: CR units or (signatures, sets))
    "CR":    ("Gronska-Peski et al. 2026, AL cerebellar neurons", U_OURS["mm10 autosomes"], U_CR),
    "mouse": ("Riva et al. 2020 mSBS", U_OURS["mm10"], (SIGS["mouse"], SIM_MOUSE_SETS)),
    "human": ("COSMIC v3.4 SBS", U_OURS["GRCh37"], (SIGS["human"], SIM_HUMAN_SETS))}
SIM_SAMPLES, _rows = {}, []
for key, (label, mine, other) in COMPARISONS.items():
    y = corrected(mine)
    row = dict(comparison=key, reference=label)
    if key == "CR":
        ref = corrected(other)
        obs = float(sim_cosine(y, ref))
        sc = sim_cosine(sample_from(ref, mine, SIM_N_SAMPLES), ref)
    else:
        sigs, sets = other
        obs, prop, best = fit_best(y, sigs, sets)
        ref = sigs[list(best)].to_numpy() @ prop
        sc = sim_cosine(sample_from(ref, mine, SIM_N_SAMPLES), ref)
        row.update(signatures=best, proportions=tuple(prop))
    n_le = int((sc <= obs).sum())
    row.update(cosine=obs, samples_mean=sc.mean(), samples_lo=np.percentile(sc, 2.5),
               samples_hi=np.percentile(sc, 97.5), samples_at_or_below=n_le,
               p=(n_le if n_le else 1) / SIM_N_SAMPLES, p_is_bound=(n_le == 0))
    SIM_SAMPLES[key] = sc
    _rows.append(row)
SIMILARITY = pd.DataFrame(_rows).set_index("comparison")

# ---------------------------------- report -----------------------------------
print(f"ours: {SIM_N_MUT:,} mutations in {len(SIM_RAW)} animals")
mix_text = lambda sigs, prop: " + ".join(f"{p:.0%} {s}" for s, p in zip(sigs, prop))
for key, r in SIMILARITY.iterrows():
    fit = f" ({mix_text(r.signatures, r.proportions)})" if key != "CR" else ""
    p = f"< {1 / SIM_N_SAMPLES:g}" if r.p_is_bound else f"= {r.p:.2g}"
    print(f"   {key:5s} {r.reference}{fit}: cosine {r.cosine:.4f}; samples {r.samples_mean:.4f} "
          f"[{r.samples_lo:.4f}, {r.samples_hi:.4f}], p {p}")

In [ ]:
# =============================================================================
# Fig. 3c: point mutations per bp in accessible and inaccessible chromatin
#
# mutations    somatic mutations (SINGLETONS_ON_MOLECULE <= 2, minimum untrimmed
#              coverage 10) of all libraries; a mutation at POS is in a region if
#              POS - 1 lies in one of its BED intervals
# callable bps in the region at minimum coverage RR_FLOOR (callable_bps_by_region_ext450.tsv.gz)
# rate         mutations / callable bps, pooled over the libraries, with an exact
#              (Garwood) 95% Poisson CI
# test         exact conditional test for two Poisson rates (given the mutations of
#              both regions, the count in the first is binomial with probability =
#              its share of the callable bps), RR_ACC_ALTERNATIVE
# =============================================================================
import os
import numpy as np
import pandas as pd
from scipy import stats

# ------------------------------ knobs ----------------------------------------
RR_MAX_SPM   = 2                       # <= this many singletons per molecule
RR_FLOOR     = 10                      # minimum untrimmed coverage of the callable bps
RR_ACC_BED   = f"{ACC}/regions/accessibility_regions.bed.gz"
RR_DENOM     = f"{ACC}/callable_bps_by_region_ext450.tsv.gz"
RR_ACC_ORDER = ["accessible", "inaccessible"]
RR_ACC_ALTERNATIVE = "greater"         # "greater" (one-sided, higher rate in accessible chromatin) or "two-sided"


# ------------------------------ helpers --------------------------------------
def label_by_bed(bed, chrom, pos_1based):
    """Region label of each 1-based position (tests pos - 1); 'none' outside every interval."""
    b = pd.read_csv(bed, sep="\t", header=None, usecols=[0, 1, 2, 3],
                    names=["chrom", "start", "end", "region"], dtype={"chrom": str})
    out = np.full(len(chrom), "none", dtype=object)
    for ch, g in b.sort_values(["chrom", "start"]).groupby("chrom"):
        s, e, r = g.start.to_numpy(), g.end.to_numpy(), g.region.to_numpy(object)
        k = np.flatnonzero(chrom == ch)
        p = pos_1based[k] - 1
        i = np.searchsorted(s, p, side="right") - 1
        ok = i >= 0
        ok[ok] = p[ok] < e[i[ok]]
        out[k[ok]] = r[i[ok]]
    return out


def garwood_ci(n, alpha=0.05):
    """Exact (Garwood) CI for a Poisson count."""
    lo = stats.chi2.ppf(alpha / 2, 2 * n) / 2 if n > 0 else 0.0
    return lo, stats.chi2.ppf(1 - alpha / 2, 2 * n + 2) / 2


def two_rates_test(na, da, nb, db, alternative="two-sided"):
    """Exact conditional test for two Poisson rates na/da and nb/db. Returns the rate
    ratio, its exact two-sided 95% CI and the p for `alternative`."""
    n, q = int(na + nb), da / (da + db)
    pl, ph = stats.binomtest(int(na), n, q).proportion_ci(0.95, method="exact")
    p = stats.binomtest(int(na), n, q, alternative=alternative).pvalue
    odds = db / da
    return (na / da) / (nb / db), pl / (1 - pl) * odds, ph / (1 - ph) * odds, p


def rate_table(N, D, order):
    """Pooled rate per region with its exact 95% CI; N, D are library x region."""
    t = pd.DataFrame({"n": N[order].sum(), "depth": D[order].sum()})
    t["rate"] = t.n / t.depth
    ci = [garwood_ci(n) for n in t.n]
    t["lo"] = [lo / d for (lo, _), d in zip(ci, t.depth)]
    t["hi"] = [hi / d for (_, hi), d in zip(ci, t.depth)]
    t["depth_share"] = t.depth / t.depth.sum()
    return t


# ---------------------- mutations and callable bps ---------------------------
_rc = pd.read_csv(f"{CAND}/somatic_mutation_candidates.tsv", sep="\t",
                  usecols=["LIBRARY", "CHROM", "POS", "SINGLETONS_ON_MOLECULE"])
_rc = _rc[_rc.SINGLETONS_ON_MOLECULE <= RR_MAX_SPM].copy()
_rc["accessibility"] = label_by_bed(RR_ACC_BED, _rc.CHROM.to_numpy(), _rc.POS.to_numpy())
_rd = pd.read_csv(RR_DENOM, sep="\t")
_rd = _rd[_rd.floor == RR_FLOOR]

D_acc = _rd[_rd.accessibility.isin(RR_ACC_ORDER)].groupby(["sample", "accessibility"]).callable_depth.sum().unstack()
D_acc = D_acc.reindex(columns=RR_ACC_ORDER)
N_acc = _rc[_rc.accessibility.isin(RR_ACC_ORDER)].groupby(["LIBRARY", "accessibility"]).size().unstack(fill_value=0)
N_acc = N_acc.reindex(index=D_acc.index, columns=RR_ACC_ORDER, fill_value=0)

acc_tab = rate_table(N_acc, D_acc, RR_ACC_ORDER)
_a, _b = RR_ACC_ORDER
*acc_rr, acc_p = two_rates_test(acc_tab.n[_a], acc_tab.depth[_a], acc_tab.n[_b], acc_tab.depth[_b],
                                RR_ACC_ALTERNATIVE)
REGION_RATES = {"accessibility": dict(table=acc_tab, p=acc_p, rr=acc_rr)}

print(f"{'region':>14s} {'mutations':>9s} {'callable bps':>16s} {'rate [95% CI]':>30s}")
for r, v in acc_tab.iterrows():
    print(f"{r:>14s} {int(v.n):>9,} {int(v.depth):>16,} {v.rate:.3e} [{v.lo:.2e}, {v.hi:.2e}]")
print(f"accessible / inaccessible rate ratio {acc_rr[0]:.2f} [{acc_rr[1]:.2f}, {acc_rr[2]:.2f}], "
      f"p = {acc_p:.2g} ({RR_ACC_ALTERNATIVE})")

In [ ]:
# =============================================================================
# Fig. 3: spectrum (a), cosine similarity to three references with the cosines of
# the samples drawn from each in small panels (b), point mutations per bp by
# chromatin accessibility (c)
#
# Layout: FIGSIZE sets the figure; a spans the width, b and c share the row below.
# Margins and gaps are in inches; A_SHARE splits the height, ROW_SHARES the width of
# the lower row. The PDF is cropped to its content (FIG_PAD_IN border).
# =============================================================================
FIGSIZE = (6.5, 4.06)               # width, height in inches
B_INSET_SIZE_IN = (0.62, 0.48)      # width, height of the small panels in b, inches

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FormatStrFormatter, MaxNLocator

plt.rcParams["pdf.fonttype"] = 42           # TrueType fonts in the PDF

# ------------------------------ layout knobs ---------------------------------
FIG_MARGINS_IN = dict(left=0.5, right=0.12, top=0.42, bottom=0.42)  # inches around the panels
AB_GAP_IN    = 0.80                 # a's bottom spine to the top spines of b and c
ROW_GAP_IN   = 0.55                 # b to c
A_SHARE      = 0.34                 # a's share of the panels' height
ROW_SHARES   = (0.72, 0.28)         # widths of b and c, as shares of the lower row
A_TITLE_IN   = 0.30                 # a's title, inches above its top spine
ROW_TITLE_IN = 0.17                 # b and c titles, inches above their top spines
FIG_PAD_IN   = 0.02                 # white border kept when the saved figure is cropped
HIDE_SPINES  = ("top", "right")     # spines hidden on b, c and b's small panels

# ------------------------------ text knobs -----------------------------------
TITLES = {"a": "96-type mutational spectrum",
          "b": "Similarity to other mutational spectra",
          "c": "Chromatin accessibility"}
FONT_TITLE, FONT_LABEL, FONT_TICK, FONT_SMALL = 9, 8, 7, 6      # titles, axis labels, tick labels, small panels
FONT_CATEGORY = max(FONT_LABEL, FONT_TICK)                       # category labels of b and c

# ------------------------------ a --------------------------------------------
A_YPAD     = 1.18                   # headroom above the tallest bar, for the class bands
A_CTX_FONT = 4                      # the 96 context labels
A_XLABEL_PAD = 3                    # space between the context labels and a's x label, points

# ------------------------------ b --------------------------------------------
B_ORDER  = ["CR", "mouse", "human"]
B_LABELS = {"CR":    "Mouse cerebellar\nneurons",          # {sigs}: the fitted signatures, "A + B"
            "mouse": "Mouse signatures\n{sigs}",
            "human": "Human signatures\n{sigs}"}
B_STYLE  = {"CR":    ("#D55E00", "s"),          # colour, marker
            "mouse": ("#CC79A7", "v"),
            "human": ("#009E73", "D")}
B_YLABEL = "Cosine similarity"
B_MS         = 5.5                  # marker size, points
B_LEFT_IN, B_RIGHT_IN = 0.45, 0.05  # left spine to the first point; last small panel to the right spine
B_Y_PAD      = 0.01                 # cosine units below the lowest point
B_TOP_PAD_IN = 0.05                 # highest small panel to the top spine
B_INSET_DX_IN, B_INSET_DY_IN = 0.12, 0.17   # small panel's lower-left corner from its point, inches
B_INSET_BIN  = 0.001                # histogram bin width, cosine units
B_INSET_TICKS = (0.92, 0.95, 0.98)  # cosine ticks of the small panels
B_OBS_COLOR  = "#4C72B0"            # our cosine in the small panels

# ------------------------------ c --------------------------------------------
C_COLOR, C_MARKER, C_MS = "#4C72B0", "o", 5
C_ELW, C_CAPSIZE = 1.0, 2.5
C_YLABEL  = "Point mutations per bp (×10$^{-8}$)"
C_Y_UNIT  = 1e-8
C_Y_NBINS = 5                       # about this many y ticks; the y axis starts at 0
C_TOP_PAD_IN = 0.05                 # p text to the top spine
C_NAMES  = {"accessible": "Accessible", "inaccessible": "Inaccessible"}
C_XLABEL = "Chromatin"
C_BR_LW, C_BR_TICK = 0.9, 0.03      # bracket line width; end ticks, as a share of the y top

SAVE_AS = "spectrum_similarity_accessibility.pdf"

# ------------------------------ helpers --------------------------------------
COSMIC_COLORS = {"C>A": "#02BCED", "C>G": "#010101", "C>T": "#E32926",
                 "T>A": "#CAC9C9", "T>C": "#A1CE63", "T>G": "#EDC6C5"}


def spectrum_panel(ax, y, ymax, ctx_font=A_CTX_FONT):
    """96 bars in COSMIC colours, class bands above the axes, context labels below."""
    ax.bar(range(96), y, width=0.78, color=[COSMIC_COLORS[c[2:5]] for c in SIM_CHANNELS], linewidth=0)
    ax.set_ylim(0, ymax)
    ax.set_xlim(-0.8, 95.8)
    ax.set_ylabel("fraction of mutations", fontsize=FONT_LABEL)
    ax.tick_params(axis="y", labelsize=FONT_TICK)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for j, sub in enumerate(COSMIC_COLORS):
        ax.add_patch(Rectangle((j * 16 - .5, ymax * 1.005), 16, ymax * .055,
                               color=COSMIC_COLORS[sub], clip_on=False, zorder=3))
        ax.text(j * 16 + 7.5, ymax * 1.065, sub, ha="center", va="bottom",
                fontsize=FONT_LABEL, fontweight="bold", clip_on=False)
    ax.set_xticks(range(96))
    ax.set_xticklabels([c[0] + c[6] for c in SIM_CHANNELS], fontsize=ctx_font,
                       family="monospace", rotation=90)
    ax.set_xlabel("trinucleotide context (5' and 3' base)", fontsize=FONT_LABEL, labelpad=A_XLABEL_PAD)


def panel_title(ax, letter, text, y, fontsize=FONT_TITLE, gap_in=0.06):
    """Bold letter + regular text, starting at the left spine, at height y (axes units)."""
    t = ax.text(0, y, letter, transform=ax.transAxes, ha="left", va="baseline",
                fontsize=fontsize, fontweight="bold")
    ax.figure.canvas.draw()
    bb = t.get_window_extent(renderer=ax.figure.canvas.get_renderer())
    dx = bb.width / (ax.get_position().width * ax.figure.get_figwidth() * ax.figure.dpi)
    gap = gap_in / (ax.get_position().width * ax.figure.get_figwidth())
    return ax.text(dx + gap, y, text, transform=ax.transAxes, ha="left", va="baseline", fontsize=fontsize)


def lighten(color, alpha):
    """color laid over white at this opacity, as an opaque colour."""
    return tuple(1 - alpha * (1 - c) for c in to_rgb(color))


def format_p(p):
    """p = 0.016 style down to 0.001, then p = 3.5 x 10^-4."""
    if p >= 1e-3:
        return f"p = {p:.2g}"
    m, e = f"{p:.1e}".split("e")
    return rf"p = {m} × 10$^{{{int(e)}}}$"


def overlaps(a, b):
    """True if two display-space bounding boxes overlap."""
    return a.x0 < b.x1 and b.x0 < a.x1 and a.y0 < b.y1 and b.y0 < a.y1


def b_label(key, r):
    """Category label of panel b, filled from the fitted signatures."""
    if key == "CR":
        return B_LABELS[key]
    return B_LABELS[key].format(sigs=" + ".join(r.signatures))


# ------------------------------ geometry -------------------------------------
W, H = FIGSIZE
M = FIG_MARGINS_IN
B_INSET_W_IN, B_INSET_H_IN = B_INSET_SIZE_IN
panels_h = H - M["top"] - M["bottom"] - AB_GAP_IN
a_h = A_SHARE * panels_h
r_h = panels_h - a_h
a_w = W - M["left"] - M["right"]
row_w = a_w - ROW_GAP_IN
b_w, c_w = (s / sum(ROW_SHARES) * row_w for s in ROW_SHARES)
assert min(a_h, r_h, b_w, c_w) > 0, f"FIGSIZE {FIGSIZE} is too small for the margins and gaps"
fig = plt.figure(figsize=FIGSIZE)
axA = fig.add_axes([M["left"] / W, (M["bottom"] + r_h + AB_GAP_IN) / H, a_w / W, a_h / H])
axB = fig.add_axes([M["left"] / W, M["bottom"] / H, b_w / W, r_h / H])
axC = fig.add_axes([(M["left"] + b_w + ROW_GAP_IN) / W, M["bottom"] / H, c_w / W, r_h / H])
row_title_y = 1 + ROW_TITLE_IN / r_h

# ------------------------------ a --------------------------------------------
_y = SPECTRUM / SPECTRUM.sum()
spectrum_panel(axA, _y, _y.max() * A_YPAD)
panel_title(axA, "a", TITLES["a"], 1 + A_TITLE_IN / a_h)

# ------------------------------ b --------------------------------------------
S = SIMILARITY.loc[B_ORDER]
n_b = len(B_ORDER)
_u = (b_w - B_LEFT_IN - B_INSET_DX_IN - B_INSET_W_IN - B_RIGHT_IN) / (n_b - 1)   # inches between references
assert _u > B_INSET_DX_IN + B_INSET_W_IN, \
    "panel b is too narrow for its small panels -- widen ROW_SHARES[0] or FIGSIZE, or shrink B_INSET_SIZE_IN"
xb = np.arange(n_b, dtype=float)
axB.set_xlim(-B_LEFT_IN / _u, (n_b - 1) + (B_INSET_DX_IN + B_INSET_W_IN + B_RIGHT_IN) / _u)
for x, (key, r) in zip(xb, S.iterrows()):
    col, mk = B_STYLE[key]
    axB.plot(x, r.cosine, marker=mk, color=col, ms=B_MS, mew=0, ls="none", zorder=3)
_y_bot = S.cosine.min() - B_Y_PAD
_h = B_INSET_DY_IN + B_INSET_H_IN + B_TOP_PAD_IN          # inches each small panel needs above its point
assert _h < r_h, "the small panels are taller than panel b -- shrink B_INSET_SIZE_IN or B_INSET_DY_IN"
_y_top = ((S.cosine * r_h - _h * _y_bot) / (r_h - _h)).max()
axB.set_ylim(_y_bot, _y_top)
_t = MaxNLocator(nbins=5, steps=[1, 2, 5, 10]).tick_values(_y_bot, _y_top)
axB.yaxis.set_major_locator(FixedLocator(_t[(_t >= _y_bot) & (_t <= min(1.0, _y_top))]))
axB.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
axB.set_xticks(xb)
axB.set_xticklabels([b_label(key, r) for key, r in S.iterrows()], fontsize=FONT_CATEGORY)
axB.set_ylabel(B_YLABEL, fontsize=FONT_LABEL)
panel_title(axB, "b", TITLES["b"], row_title_y)
_insets = []
_all = np.concatenate([SIM_SAMPLES[k] for k in B_ORDER] + [S.cosine.to_numpy()])
_common = (_all.min() - 0.04 * np.ptp(_all), _all.max() + 0.04 * np.ptp(_all))
_common = (min(_common[0], min(B_INSET_TICKS) - 0.005), max(_common[1], max(B_INSET_TICKS) + 0.005))
for x, (key, r) in zip(xb, S.iterrows()):
    col = B_STYLE[key][0]
    px, py = axB.transData.transform((x, r.cosine)) / fig.dpi             # the point, inches
    ins = fig.add_axes([(px + B_INSET_DX_IN) / W, (py + B_INSET_DY_IN) / H, B_INSET_W_IN / W, B_INSET_H_IN / H])
    s = SIM_SAMPLES[key]
    edges = np.arange(np.floor(s.min() / B_INSET_BIN) * B_INSET_BIN,
                      np.ceil(s.max() / B_INSET_BIN) * B_INSET_BIN + B_INSET_BIN / 2, B_INSET_BIN)
    freq = np.histogram(s, edges)[0] / len(s)
    ins.stairs(freq, edges, fill=True, facecolor=lighten(col, 0.35), edgecolor=col, linewidth=0.7)
    ins.axvline(r.cosine, color=B_OBS_COLOR, lw=1.0)
    ins.set_xlim(*_common)
    ins.set_ylim(0, freq.max() * 1.15)
    ins.set_yticks([])
    ins.xaxis.set_major_locator(FixedLocator(B_INSET_TICKS))
    ins.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ins.tick_params(axis="x", labelsize=FONT_SMALL, length=2, width=0.6, pad=1)
    for name, sp in ins.spines.items():
        sp.set_linewidth(0.6)
        sp.set_visible(name not in HIDE_SPINES)
    _insets.append(ins)

# ------------------------------ c --------------------------------------------
_acc = REGION_RATES["accessibility"]
tc = _acc["table"].loc[RR_ACC_ORDER]
xc = np.arange(len(RR_ACC_ORDER), dtype=float)
_eb = axC.errorbar(xc, tc.rate / C_Y_UNIT, yerr=[(tc.rate - tc.lo) / C_Y_UNIT, (tc.hi - tc.rate) / C_Y_UNIT],
                   fmt=C_MARKER, ms=C_MS, color=C_COLOR, ecolor=C_COLOR, elinewidth=C_ELW,
                   capsize=C_CAPSIZE, capthick=C_ELW, zorder=3)
_eb.lines[0].set_markeredgewidth(0)
_hc = tc.hi.max() / C_Y_UNIT
_y_br = 1.06 * _hc                                         # bracket over the pair
_tick = C_BR_TICK * _hc
axC.plot([xc[0], xc[0], xc[1], xc[1]], [_y_br - _tick, _y_br, _y_br, _y_br - _tick], color=C_COLOR,
         lw=C_BR_LW, solid_capstyle="butt", zorder=4)
_y_p = _y_br + 0.4 * _tick
_p_txt = axC.text(xc.mean(), _y_p, format_p(_acc["p"]), ha="center", va="bottom", fontsize=FONT_LABEL,
                  color=C_COLOR)
axC.set_ylim(0, 2 * _y_p)                                  # provisional; the top is set below
fig.canvas.draw()
_ren = fig.canvas.get_renderer()
_hp = (_p_txt.get_window_extent(_ren).y1 - axC.transData.transform((0, _y_p))[1]) / fig.dpi + C_TOP_PAD_IN
axC.set_ylim(0, _y_p * r_h / (r_h - _hp))                  # p text C_TOP_PAD_IN under the top spine
axC.yaxis.set_major_locator(MaxNLocator(nbins=C_Y_NBINS, steps=[1, 2, 2.5, 5, 10]))
axC.set_ylabel(C_YLABEL, fontsize=FONT_LABEL)
axC.set_xlim(-0.5, len(RR_ACC_ORDER) - 0.5)
axC.set_xticks(xc)
axC.set_xticklabels([C_NAMES[g] for g in RR_ACC_ORDER], fontsize=FONT_CATEGORY)
axC.set_xlabel(C_XLABEL, fontsize=FONT_LABEL)
panel_title(axC, "c", TITLES["c"], row_title_y)
for ax in (axB, axC):
    ax.tick_params(axis="y", labelsize=FONT_TICK)
    ax.tick_params(axis="x", labelsize=FONT_CATEGORY)
    ax.grid(False)
    for name in HIDE_SPINES:
        ax.spines[name].set_visible(False)

# ------------------------------ checks ---------------------------------------
fig.canvas.draw()
_ren = fig.canvas.get_renderer()
for ax in fig.axes:
    _bb = ax.get_tightbbox(_ren)
    assert _bb.x0 >= 0 and _bb.y0 >= 0 and _bb.x1 <= fig.bbox.x1 and _bb.y1 <= fig.bbox.y1, \
        "text runs off the figure -- widen FIG_MARGINS_IN or FIGSIZE"
_ib = [ins.get_tightbbox(_ren) for ins in _insets]
for i in range(len(_ib)):
    for j in range(i + 1, len(_ib)):
        if overlaps(_ib[i], _ib[j]):
            print(f"note: the {B_ORDER[i]} and {B_ORDER[j]} small panels overlap -- widen panel b or shrink B_INSET_SIZE_IN")

os.makedirs(FIG_DIR, exist_ok=True)
path = os.path.join(FIG_DIR, SAVE_AS)
fig.savefig(path, bbox_inches="tight", pad_inches=FIG_PAD_IN)     # cropped; the panels keep their size
plt.show()
print(f"-> {path}")